# Image Classification - Fashion MNIST

## Problem Statement
Classify images of clothing items into 10 distinct categories using a Convolutional Neural Network (CNN).
We are using the **Fashion MNIST** dataset provided in `fashion-mnist_test.csv`.

### Dataset Overview
- **Classes**: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.
- **Input**: 28x28 grayscale images.
- **Source**: Single CSV file containing 10,000 samples.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Set random seed
tf.random.set_seed(42)
np.random.seed(42)


## 1. Data Loading and Preprocessing
We load the data from the CSV file. The file typically contains a 'label' column and 784 pixel columns (pixel1 to pixel784).


In [ ]:
# Load dataset
df = pd.read_csv('fashion-mnist_test.csv')
print(f"Dataset Shape: {df.shape}")
df.head()


### Separating Labels and Images
We extract the labels and the pixel data. Then we reshape the flat 784-pixel vectors into 28x28x1 images.


In [ ]:
# Extract labels and pixels
labels = df['label'].values
pixels = df.drop('label', axis=1).values

# Reshape to (N, 28, 28, 1)
images = pixels.reshape(-1, 28, 28, 1)

# Normalize pixel values to [0, 1]
images = images.astype('float32') / 255.0

print(f"Images Shape: {images.shape}")
print(f"Labels Shape: {labels.shape}")


### Splitting Data
Since we have a single file of 10,000 samples, we will split it into training (80%) and testing (20%) sets.


In [ ]:
train_images, test_images, train_labels, test_labels = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training set: {train_images.shape}")
print(f"Test set: {test_images.shape}")


### Visual Inspection
Let's define the class names and visualize some samples.


In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    # Reshape back to 28x28 for display
    plt.imshow(train_images[i].reshape(28,28), cmap='gray')
    plt.xlabel(class_names[train_labels[i]])
plt.show()


## 2. Build CNN Architecture
We will build a CNN appropriate for 28x28 grayscale images.
- **Conv2D**: Feature extraction.
- **MaxPooling2D**: Downsampling.
- **Flatten**: Connecting conv layers to dense layers.
- **Dense**: Classification.


In [ ]:
def create_model():
    model = models.Sequential([
        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        
        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Flatten and Dense
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10) # 10 classes
    ])
    return model

model = create_model()
model.summary()


## 3. Train the Model
We compile with Adam optimizer and train for 15 epochs.


In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=15, 
                    validation_data=(test_images, test_labels))


## 4. Evaluation and Visualization
Plotting accuracy and loss.


In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.show()

plot_history(history)


### Performance on Test Set


In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")


### Visualizing Feature Maps
Visualizing the output of the first convolutional layer.


In [ ]:
feature_map_model = models.Model(inputs=model.inputs, outputs=model.layers[0].output)
img = test_images[0:1]
feature_maps = feature_map_model.predict(img)

plt.figure(figsize=(16, 8))
for i in range(32):
    plt.subplot(4, 8, i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap='viridis')
    plt.axis('off')
plt.suptitle('Feature Maps (Conv Layer 1)')
plt.show()


### Misclassified Images


In [ ]:
predictions = model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)
misclassified_indices = np.where(predicted_labels != test_labels)[0]

plt.figure(figsize=(12, 8))
for i, idx in enumerate(misclassified_indices[:15]):
    plt.subplot(3, 5, i+1)
    plt.imshow(test_images[idx].reshape(28,28), cmap='gray')
    plt.title(f"True: {class_names[test_labels[idx]]}\nPred: {class_names[predicted_labels[idx]]}", 
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


## 5. Model Improvement
Adding Dropout and Data Augmentation.


In [ ]:
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal", input_shape=(28, 28, 1)),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

def create_improved_model():
    model = models.Sequential([
        data_augmentation,
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10)
    ])
    return model

improved_model = create_improved_model()
improved_model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history_improved = improved_model.fit(train_images, train_labels, epochs=20, # More epochs for aug
                                      validation_data=(test_images, test_labels))


In [ ]:
plot_history(history_improved)
test_loss_imp, test_acc_imp = improved_model.evaluate(test_images, test_labels, verbose=2)
print(f"\nImproved Model Accuracy: {test_acc_imp:.4f}")


## Sample Predictions


In [ ]:
probability_model = tf.keras.Sequential([improved_model, tf.keras.layers.Softmax()])
predictions = probability_model.predict(test_images[:10])

plt.figure(figsize=(15, 3))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(test_images[i].reshape(28,28), cmap='gray')
    predicted_label = np.argmax(predictions[i])
    true_label = test_labels[i]
    color = 'green' if predicted_label == true_label else 'red'
    plt.xlabel(f"{class_names[predicted_label]}\n({100*np.max(predictions[i]):.0f}%)", color=color)
    plt.xticks([])
    plt.yticks([])
plt.show()
